In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop stream if the Zoom link is private?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [9]:
truth = ground_truth[10]

In [10]:
doc_idx[truth["document"]]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [13]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join. It’s not too late to start.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [14]:
assistant.total_cost()

0.0006285

In [15]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [16]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course late — can I still join, or is it too late to start?',
 'answer_llm': 'Yes, you can still join. It’s not too late to start.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [17]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [18]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course late — can I still join, or is it too late to start?',
 'answer_llm': 'Yes — you can still join. You can start learning anytime, but if you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [19]:
assistant.reset_usage()

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [21]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [22]:
results[:10]

[{'question': 'I just found this course late — can I still join, or is it too late to start?',
  'answer_llm': 'Yes, you can still join. You can start learning anytime, but if you want a certificate, you’ll need to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to enroll if I only discovered the course now?',
  'answer_llm': 'Yes, you can still join if you just discovered the course. If you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has started, can I still get a certificate?',
  'answer_llm': 'Yes, but if you want to receive a cer

In [25]:
df_results = pd.DataFrame(results)
df_results.to_csv("data/rag-results.csv", index=False)
df_results.head(5)

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still jo...,"Yes, you can still join. You can start learnin...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Am I allowed to enroll if I only discovered th...,"Yes, you can still join if you just discovered...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join after the course has started, can I ...","Yes, but if you want to receive a certificate,...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to qualify for the certif...,"To qualify for the certificate, you need to:\n...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Is it okay to start the course now, and what’s...","Yes, you can start the course whenever you wan...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [26]:
assistant.total_cost()

0.53573775